## E2E Incremental Report Case 5 with ContainerEvent

This notebook validates incremental processing with definition hash changes while BasicEvent and ContainerEvent coexist.

In [ ]:
import os
import sys
from pyspark.sql import functions as F

module_path = os.getcwd()
module_path = "/".join(module_path.split("/")[:-2]) + "/src"
sys.path.insert(0, module_path)

from mda_reporting.aggregations.histogram import HistogramDuration
from mda_reporting.aggregations.histogram2d import Histogram2DDuration
from mda_reporting.aggregations.stats_aggregator import StatsAggregator
from mda_reporting.core.page import Page
from mda_reporting.core.report import Report
from mda_reporting.events.basic_event import BasicEvent
from mda_reporting.events.container_event import ContainerEvent

In [ ]:
BASE_CONTAINER_TABLE = "container_metric"
BASE_CHANNEL_METRICS_TABLE = "development.silver.channel_metric"
BASE_CHANNELS_URI = "development.silver.channel_data"

UNITY_CATALOG = "development"
UNITY_SCHEMA = "gold_e2e"
UNITY_PREFIX = "e2e"


def build_config(container_metrics_table: str):
    return {
        "incremental": {
            "enabled": True,
            "silver_last_modified_column": "timestamp",
            "gold_last_modified_column": "_created_at",
        },
        "source": {
            "container_metrics_table": f"development.silver.{container_metrics_table}",
            "channel_metrics_table": BASE_CHANNEL_METRICS_TABLE,
            "channels_uri": BASE_CHANNELS_URI,
        },
        "unity_sink": {
            "catalog": UNITY_CATALOG,
            "schema": UNITY_SCHEMA,
            "table_prefix": UNITY_PREFIX,
        },
        "query_engine": {"solver": "BasicNarrowSolver"},
        "measurement_dimensions": ["container_id", "start_ts", "stop_ts"],
    }


def add_aggs_to_report(report):
    query = report.get_db().query
    c1 = query.channel(channel_name="is1_eng_speed", data_key="TM")
    c2 = query.channel(channel_name="can_vehicle_speed", data_key="TM")

    first_page = Page(page_number=1)
    report.add_page(first_page)

    hist1 = HistogramDuration(
        "rpm_hist_p1", base_expr=c1, bins=[float(i) for i in range(0, 8000, 250)]
    )
    hist2 = HistogramDuration(
        "speed_hist_p1", base_expr=c2, bins=[float(i) for i in range(0, 300, 1)]
    )
    first_page.add_aggregation(hist1)
    first_page.add_aggregation(hist2)

    hist2d = Histogram2DDuration(
        name="rpm_speed_heatmap",
        x_expr=c1,
        y_expr=c2,
        x_bins=[float(i) for i in range(0, 8000, 500)],
        y_bins=[float(i) for i in range(0, 300, 25)],
        x_channel_name="is1_eng_speed",
        y_channel_name="can_vehicle_speed",
    )
    first_page.add_aggregation(hist2d)
    engine_rpm_event = BasicEvent(
        name="rpm_event",
        expr=c1 > 0,
        desc="engine speed > 0 rpm",
    )
    container_event = ContainerEvent("Measurement Event")

    stats_agg = StatsAggregator(
        name="stats_agg",
        input_expressions=[c1],
        channel_names=["Engine RPM"],
        event=engine_rpm_event,
        statistics=["start", "end", "mean"],
    )
    stats_agg_container = StatsAggregator(
        name="stats_agg_container",
        input_expressions=[c1],
        channel_names=["Engine RPM"],
        event=container_event,
        statistics=["start", "end", "mean"],
    )

    report.add_event(engine_rpm_event)
    report.add_event(container_event)
    first_page.add_aggregation(stats_agg)
    first_page.add_aggregation(stats_agg_container)


def add_aggs_to_report_changed_bins(report):
    query = report.get_db().query
    c1 = query.channel(channel_name="is1_eng_speed", data_key="TM")
    c2 = query.channel(channel_name="can_vehicle_speed", data_key="TM")

    first_page = Page(page_number=1)
    report.add_page(first_page)

    hist1 = HistogramDuration(
        "rpm_hist_p1", base_expr=c1, bins=[float(i) for i in range(0, 8000, 1)]
    )
    hist2 = HistogramDuration(
        "speed_hist_p1", base_expr=c2, bins=[float(i) for i in range(0, 300, 1)]
    )
    first_page.add_aggregation(hist1)
    first_page.add_aggregation(hist2)

    hist2d = Histogram2DDuration(
        name="rpm_speed_heatmap",
        x_expr=c1,
        y_expr=c2,
        x_bins=[float(i) for i in range(0, 8000, 250)],
        y_bins=[float(i) for i in range(0, 300, 10)],
        x_channel_name="is1_eng_speed",
        y_channel_name="can_vehicle_speed",
    )
    first_page.add_aggregation(hist2d)

    engine_rpm_event = BasicEvent(
        name="rpm_event",
        expr=c1 > 0,
        desc="engine speed > 0 rpm",
    )
    container_event = ContainerEvent("Measurement Event")

    stats_agg = StatsAggregator(
        name="stats_agg",
        input_expressions=[c1],
        channel_names=["Engine RPM"],
        event=engine_rpm_event,
        statistics=["start", "end", "mean", "median", "min", "max"],
    )
    stats_agg_container = StatsAggregator(
        name="stats_agg_container",
        input_expressions=[c1],
        channel_names=["Engine RPM"],
        event=container_event,
        statistics=["start", "end", "mean"],
    )

    report.add_event(engine_rpm_event)
    report.add_event(container_event)
    first_page.add_aggregation(stats_agg)
    first_page.add_aggregation(stats_agg_container)

In [ ]:
container_metric_df = spark.read.table(f"development.silver.{BASE_CONTAINER_TABLE}")
container_metric_count = container_metric_df.count()
print(f"{BASE_CONTAINER_TABLE} count = {container_metric_count}")
assert (
    container_metric_count == 1858
), "Expected 1858 measurement files in silver.container_metric"

sample_container_ids = [
    row.container_id
    for row in container_metric_df.select("container_id")
    .distinct()
    .orderBy("container_id")
    .limit(2)
    .collect()
]
assert (
    len(sample_container_ids) == 2
), "Expected at least two containers for pre-state setup"

pre_source_df = container_metric_df.where(
    F.col("container_id").isin(sample_container_ids)
)
pre_source_df.write.format("delta").mode("overwrite").saveAsTable(
    "development.silver.container_metric_case5_pre"
)

PRE_CONTAINER_TABLE = "container_metric_case5_pre"

print(
    f"Using pre-source table {PRE_CONTAINER_TABLE} with container_ids={sample_container_ids}"
)

In [ ]:
for tbl in spark.catalog.listTables(f"{UNITY_CATALOG}.{UNITY_SCHEMA}"):
    if tbl.name.startswith(f"{UNITY_PREFIX}_"):
        spark.sql(f"DROP TABLE IF EXISTS {UNITY_CATALOG}.{UNITY_SCHEMA}.{tbl.name}")

print("Cleared existing gold tables for this test prefix")

In [ ]:
report_pre = Report(
    name="incremental_case5_pre", spark=spark, config=build_config(PRE_CONTAINER_TABLE)
)
add_aggs_to_report(report_pre)
report_pre.determine_report()
report_pre.persist_results()

hist_dim_pre = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_histogram_dimension"
)
hist_fact_pre = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_histogram_fact"
)
stats_fact_pre = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_stats_aggregator_fact"
)

hist1_hash_pre = (
    hist_dim_pre.where(F.col("name") == "rpm_hist_p1")
    .select("definition_hash")
    .collect()[0][0]
)
hist1_visual_id = (
    hist_dim_pre.where(F.col("name") == "rpm_hist_p1")
    .select("visual_id")
    .collect()[0][0]
)
hist2_visual_id = (
    hist_dim_pre.where(F.col("name") == "speed_hist_p1")
    .select("visual_id")
    .collect()[0][0]
)

hist2_fact_pre = (
    hist_fact_pre.where(
        (F.col("visual_id") == hist2_visual_id)
        & (F.col("container_id").isin(sample_container_ids))
    )
    .orderBy("container_id", "bin_id")
    .collect()
)

total_hist_count_pre = hist_fact_pre.count()
stats_count_pre = stats_fact_pre.count()
pre_measurement_count = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_measurement_dimension"
).count()

assert pre_measurement_count == 2, "Pre-state should contain exactly two containers"

In [ ]:
stats_count_pre

In [ ]:
report_post = Report(
    name="incremental_case5_post",
    spark=spark,
    config=build_config(BASE_CONTAINER_TABLE),
)
add_aggs_to_report_changed_bins(report_post)
report_post.determine_report()
report_post.persist_results()

measurement_dim_post = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_measurement_dimension"
)
hist_dim_post = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_histogram_dimension"
)
hist_fact_post = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_histogram_fact"
)
stats_dim_post = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_stats_aggregator_dimension"
)
stats_fact_post = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_stats_aggregator_fact"
)
event_dim_post = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_event_dimension"
)

In [ ]:
post_measurement_count = measurement_dim_post.count()
assert post_measurement_count == 1858, "Expected 1858 measurements in post-state"

hist1_hash_post = (
    hist_dim_post.where(F.col("name") == "rpm_hist_p1")
    .select("definition_hash")
    .collect()[0][0]
)
assert hist1_hash_post != hist1_hash_pre, "rpm_hist_p1 definition hash must change"

hist2_fact_post = (
    hist_fact_post.where(F.col("visual_id") == hist2_visual_id)
    .orderBy("container_id", "bin_id")
    .collect()
)
assert len(hist2_fact_post) > len(
    hist2_fact_pre
), "speed_hist_p1 must increase in post-state"

total_hist_count_post = hist_fact_post.count()
stats_count_post = stats_fact_post.count()
assert (
    total_hist_count_post > total_hist_count_pre
), "Histogram fact rows should increase in post-state"
assert (
    stats_count_post > stats_count_pre
), "Stats aggregator fact rows should increase in post-state"

event_types = {
    row.event_type for row in event_dim_post.select("event_type").distinct().collect()
}
assert "BASIC_EVENT" in event_types
assert "CONTAINER_EVENT" in event_types

stats_names = {row.name for row in stats_dim_post.select("name").distinct().collect()}
assert "stats_agg" in stats_names
assert "stats_agg_container" in stats_names

joined_stats_events = stats_fact_post.join(event_dim_post, on="event_id", how="inner")
joined_count = joined_stats_events.count()
assert (
    joined_count > 0
), "Stats fact inner joined with event dimension must not be empty"

print("All assertions passed for incremental case 5 with BasicEvent + ContainerEvent.")
print(f"Post measurement count: {post_measurement_count}")
print(f"Joined stats/event rows: {joined_count}")

In [ ]:
stats_count_post

## E2E Incremental Report Case 5 with ContainerEvent and SequenceOfEvents

This notebook validates incremental processing with definition hash changes while `BasicEvent`, `ContainerEvent`, and `SequenceOfEvents(max_overlap=...)` coexist.

In [ ]:
import os
import sys
from pyspark.sql import functions as F

module_path = os.getcwd()
module_path = "/".join(module_path.split("/")[:-2]) + "/src"
sys.path.insert(0, module_path)

from mda_reporting.aggregations.histogram import HistogramDuration
from mda_reporting.aggregations.histogram2d import Histogram2DDuration
from mda_reporting.aggregations.stats_aggregator import StatsAggregator
from mda_reporting.core.page import Page
from mda_reporting.core.report import Report
from mda_reporting.events.basic_event import BasicEvent
from mda_reporting.events.container_event import ContainerEvent
from mda_reporting.events.sequence_of_events import SequenceOfEvents

In [ ]:
BASE_CONTAINER_TABLE = "container_metric"
BASE_CHANNEL_METRICS_TABLE = "development.silver.channel_metric"
BASE_CHANNELS_URI = "development.silver.channel_data"

UNITY_CATALOG = "development"
UNITY_SCHEMA = "gold_e2e"
UNITY_PREFIX = "e2e"


def build_config(container_metrics_table: str):
    return {
        "incremental": {
            "enabled": True,
            "silver_last_modified_column": "timestamp",
            "gold_last_modified_column": "_created_at",
        },
        "source": {
            "container_metrics_table": f"development.silver.{container_metrics_table}",
            "channel_metrics_table": BASE_CHANNEL_METRICS_TABLE,
            "channels_uri": BASE_CHANNELS_URI,
        },
        "unity_sink": {
            "catalog": UNITY_CATALOG,
            "schema": UNITY_SCHEMA,
            "table_prefix": UNITY_PREFIX,
        },
        "units_under_test": [],
        "query_engine": {"solver": "BasicNarrowSolver"},
        "measurement_dimensions": ["container_id", "start_ts", "stop_ts"],
    }


def add_aggs_to_report(report):
    query = report.get_db().query
    c1 = query.channel(channel_name="is1_eng_speed", data_key="TM")
    c2 = query.channel(channel_name="can_vehicle_speed", data_key="TM")

    first_page = Page(page_number=1)
    report.add_page(first_page)

    hist1 = HistogramDuration(
        "rpm_hist_p1", base_expr=c1, bins=[float(i) for i in range(0, 8000, 250)]
    )
    hist2 = HistogramDuration(
        "speed_hist_p1", base_expr=c2, bins=[float(i) for i in range(0, 300, 1)]
    )
    first_page.add_aggregation(hist1)
    first_page.add_aggregation(hist2)

    hist2d = Histogram2DDuration(
        name="rpm_speed_heatmap",
        x_expr=c1,
        y_expr=c2,
        x_bins=[float(i) for i in range(0, 8000, 500)],
        y_bins=[float(i) for i in range(0, 300, 25)],
        x_channel_name="is1_eng_speed",
        y_channel_name="can_vehicle_speed",
    )
    first_page.add_aggregation(hist2d)

    engine_rpm_event = BasicEvent(
        name="rpm_event",
        expr=c1 > 0,
        desc="engine speed > 0 rpm",
    )
    container_event = ContainerEvent("Measurement Event")
    sequence_event = SequenceOfEvents(
        name="speed_sequence_event",
        expressions=[
            (c2 > 10) & (c2 < 20),
            (c2 > 9) & (c2 < 25),
        ],
        desc="speed overlap sequence with max overlap guard",
        max_overlap=500.0,
    )

    stats_agg = StatsAggregator(
        name="stats_agg",
        input_expressions=[c1],
        channel_names=["Engine RPM"],
        event=engine_rpm_event,
        statistics=["start", "end", "mean"],
    )
    stats_agg_container = StatsAggregator(
        name="stats_agg_container",
        input_expressions=[c1],
        channel_names=["Engine RPM"],
        event=container_event,
        statistics=["start", "end", "mean"],
    )

    report.add_event(engine_rpm_event)
    report.add_event(container_event)
    report.add_event(sequence_event)
    first_page.add_aggregation(stats_agg)
    first_page.add_aggregation(stats_agg_container)


def add_aggs_to_report_changed_bins(report):
    query = report.get_db().query
    c1 = query.channel(channel_name="is1_eng_speed", data_key="TM")
    c2 = query.channel(channel_name="can_vehicle_speed", data_key="TM")

    first_page = Page(page_number=1)
    report.add_page(first_page)

    hist1 = HistogramDuration(
        "rpm_hist_p1", base_expr=c1, bins=[float(i) for i in range(0, 8000, 1)]
    )
    hist2 = HistogramDuration(
        "speed_hist_p1", base_expr=c2, bins=[float(i) for i in range(0, 300, 1)]
    )
    first_page.add_aggregation(hist1)
    first_page.add_aggregation(hist2)

    hist2d = Histogram2DDuration(
        name="rpm_speed_heatmap",
        x_expr=c1,
        y_expr=c2,
        x_bins=[float(i) for i in range(0, 8000, 250)],
        y_bins=[float(i) for i in range(0, 300, 10)],
        x_channel_name="is1_eng_speed",
        y_channel_name="can_vehicle_speed",
    )
    first_page.add_aggregation(hist2d)

    engine_rpm_event = BasicEvent(
        name="rpm_event",
        expr=c1 > 0,
        desc="engine speed > 0 rpm",
    )
    container_event = ContainerEvent("Measurement Event")
    sequence_event = SequenceOfEvents(
        name="speed_sequence_event",
        expressions=[
            (c2 > 10) & (c2 < 20),
            (c2 > 9) & (c2 < 35),
        ],
        desc="speed overlap sequence with max overlap guard",
        max_overlap=500.0,
    )

    stats_agg = StatsAggregator(
        name="stats_agg",
        input_expressions=[c1],
        channel_names=["Engine RPM"],
        event=engine_rpm_event,
        statistics=["start", "end", "mean", "median", "min", "max"],
    )
    stats_agg_container = StatsAggregator(
        name="stats_agg_container",
        input_expressions=[c1],
        channel_names=["Engine RPM"],
        event=container_event,
        statistics=["start", "end", "mean"],
    )

    report.add_event(engine_rpm_event)
    report.add_event(container_event)
    report.add_event(sequence_event)
    first_page.add_aggregation(stats_agg)
    first_page.add_aggregation(stats_agg_container)

In [ ]:
for tbl in spark.catalog.listTables(f"{UNITY_CATALOG}.{UNITY_SCHEMA}"):
    if tbl.name.startswith(f"{UNITY_PREFIX}_"):
        spark.sql(f"DROP TABLE IF EXISTS {UNITY_CATALOG}.{UNITY_SCHEMA}.{tbl.name}")

print("Cleared existing gold tables for this test prefix")

In [ ]:
container_metric_df = spark.read.table(f"development.silver.{BASE_CONTAINER_TABLE}")
container_metric_count = container_metric_df.count()
print(f"{BASE_CONTAINER_TABLE} count = {container_metric_count}")
assert (
    container_metric_count == 9
), "Expected 9 measurement files in silver.container_metric"

sample_container_ids = [
    row.container_id
    for row in container_metric_df.select("container_id")
    .distinct()
    .orderBy("container_id")
    .limit(2)
    .collect()
]
assert (
    len(sample_container_ids) == 2
), "Expected at least two containers for pre-state setup"

pre_source_df = container_metric_df.where(
    F.col("container_id").isin(sample_container_ids)
)
pre_source_df.write.format("delta").mode("overwrite").saveAsTable(
    "development.silver.container_metric_case5_pre"
)

PRE_CONTAINER_TABLE = "container_metric_case5_pre"

print(
    f"Using pre-source table {PRE_CONTAINER_TABLE} with container_ids={sample_container_ids}"
)

In [ ]:
report_pre = Report(
    name="incremental_case5_pre", spark=spark, config=build_config(PRE_CONTAINER_TABLE)
)
add_aggs_to_report(report_pre)
report_pre.determine_report()
report_pre.persist_results()

hist_dim_pre = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_histogram_dimension"
)
hist_fact_pre = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_histogram_fact"
)
stats_fact_pre = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_stats_aggregator_fact"
)

hist1_hash_pre = (
    hist_dim_pre.where(F.col("name") == "rpm_hist_p1")
    .select("definition_hash")
    .collect()[0][0]
)
hist1_visual_id = (
    hist_dim_pre.where(F.col("name") == "rpm_hist_p1")
    .select("visual_id")
    .collect()[0][0]
)
hist2_visual_id = (
    hist_dim_pre.where(F.col("name") == "speed_hist_p1")
    .select("visual_id")
    .collect()[0][0]
)

hist2_fact_pre = (
    hist_fact_pre.where(
        (F.col("visual_id") == hist2_visual_id)
        & (F.col("container_id").isin(sample_container_ids))
    )
    .orderBy("container_id", "bin_id")
    .collect()
)

total_hist_count_pre = hist_fact_pre.count()
stats_count_pre = stats_fact_pre.count()
pre_measurement_count = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_measurement_dimension"
).count()

assert pre_measurement_count == 2, "Pre-state should contain exactly two containers"

In [ ]:
stats_count_pre

In [ ]:
report_post = Report(
    name="incremental_case5_post",
    spark=spark,
    config=build_config(BASE_CONTAINER_TABLE),
)
add_aggs_to_report_changed_bins(report_post)
report_post.determine_report()
report_post.persist_results()

measurement_dim_post = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_measurement_dimension"
)
hist_dim_post = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_histogram_dimension"
)
hist_fact_post = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_histogram_fact"
)
stats_dim_post = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_stats_aggregator_dimension"
)
stats_fact_post = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_stats_aggregator_fact"
)
event_dim_post = spark.read.table(
    f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_event_dimension"
)

In [ ]:
post_measurement_count = measurement_dim_post.count()
assert post_measurement_count == 9, "Expected 9 measurements in post-state"

hist1_hash_post = (
    hist_dim_post.where(F.col("name") == "rpm_hist_p1")
    .select("definition_hash")
    .collect()[0][0]
)
assert hist1_hash_post != hist1_hash_pre, "rpm_hist_p1 definition hash must change"

hist2_fact_post = (
    hist_fact_post.where(F.col("visual_id") == hist2_visual_id)
    .orderBy("container_id", "bin_id")
    .collect()
)
assert len(hist2_fact_post) > len(
    hist2_fact_pre
), "speed_hist_p1 must increase in post-state"

total_hist_count_post = hist_fact_post.count()
stats_count_post = stats_fact_post.count()
assert (
    total_hist_count_post > total_hist_count_pre
), "Histogram fact rows should increase in post-state"
assert (
    stats_count_post > stats_count_pre
), "Stats aggregator fact rows should increase in post-state"

event_types = {
    row.event_type for row in event_dim_post.select("event_type").distinct().collect()
}
assert "BASIC_EVENT" in event_types
assert "CONTAINER_EVENT" in event_types
assert "SEQUENCE_OF_EVENTS" in event_types

event_names = {
    row.event_name for row in event_dim_post.select("event_name").distinct().collect()
}
assert "speed_sequence_event" in event_names

stats_names = {row.name for row in stats_dim_post.select("name").distinct().collect()}
assert "stats_agg" in stats_names
assert "stats_agg_container" in stats_names

joined_stats_events = stats_fact_post.join(event_dim_post, on="event_id", how="inner")
joined_count = joined_stats_events.count()
assert (
    joined_count > 0
), "Stats fact inner joined with event dimension must not be empty"

print(
    "All assertions passed for incremental case 5 with BasicEvent + ContainerEvent + SequenceOfEvents."
)
print(f"Post measurement count: {post_measurement_count}")
print(f"Joined stats/event rows: {joined_count}")